<div style="padding:24px;border-radius:16px;background:linear-gradient(120deg,#071a2b,#124e78);color:white">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#9fe7f5">Qompute 2026 · Track 2: Optimizing for People</div>
  <h1 style="margin:8px 0 4px 0;font-size:38px">Qommuter LA</h1>
  <div style="font-size:20px">Equitable bus-investment allocation with QAOA</div>
  <div style="margin-top:14px;color:#d9f4fa">A reproducible Los Angeles case study balancing observed ridership, corridor equity, and overlapping service.</div>
</div>

> **Central question:** When a city changes the values encoded in “optimal,” which communities receive investment?


## Executive result

This notebook selects **exactly 3 of 6 LA Metro bus routes**. The main scenario gives equity a weight of $\alpha=0.55$ and penalizes redundant corridor coverage. The exact optimizer selects **Lines 18, 51, and 204**. A depth-$p=3$ QAOA statevector simulation is compared against that exact answer across multiple random starts.

The contribution is not a claim of quantum advantage. It is a transparent demonstration that:

1. a civic allocation question can be encoded as a QUBO and Ising Hamiltonian;
2. every algebraic transformation can be tested;
3. QAOA output can be evaluated against an exact classical reference; and
4. the policy result changes when the equity weight changes.

> 🟠 **Data-readiness disclosure:** Q2 2024 weekday boardings below are observed LA Metro values. The equity and overlap inputs are **clearly labeled prototype values** because the final GTFS × Metro Equity Need Index spatial join has not yet been completed. They exercise the complete model but must be replaced and re-run before submission. No prototype value is presented as an official LA Metro statistic.


## Judge-criteria map

| Criterion | Weight | Evidence in this notebook |
|---|---:|---|
| Technical content | 35% | QUBO, hand-derived Ising coefficients, 64-state equality test, Qiskit QAOA implementation, exact baseline, depth/seed/penalty checks |
| Societal impact | 20% | Explicit equity choice, benefit distribution, proxy limitations, harms, guardrails, community-review plan |
| Presentation | 20% | Executive result, labeled charts, scenario comparison, concise interpretation after each experiment |
| Originality | 20% | Real LA routes, policy slider $\alpha$, pairwise route-overlap term, counterfactual ablation |
| Submission | 5% | Reproducible environment notes, source register, assumptions, attribution, pre-submission checklist |

### Notebook roadmap

1. Frame the allocation decision and inspect inputs  
2. Build the objective, QUBO, and Ising Hamiltonian  
3. Prove the encoding against exhaustive enumeration  
4. Compare QAOA with exact brute force  
5. Stress-test weights, penalties, depth, and interactions  
6. Discuss who benefits, what can go wrong, and what must happen next


## 1 · Decision context and scope

Suppose a one-time service-improvement fund can support exactly three corridors. Each candidate has:

- an **efficiency signal**: observed average weekday boardings;
- an **equity signal**: the degree to which its service area overlaps communities with high transportation need; and
- a **pairwise interaction**: how much two routes duplicate the same catchment or corridor.

This is a demonstration-scale model: six binary decisions fit on six qubits and allow complete classical verification. It does **not** estimate the causal ridership gain from funding, dictate a real budget, or replace public participation.

### Reproducibility

The saved outputs were produced with Python, NumPy, pandas, SciPy, and Matplotlib. The Qiskit cell uses the current `qaoa_ansatz` + `StatevectorEstimator` interface. Because Qiskit is not installed in every notebook host, a mathematically equivalent NumPy statevector implementation is included as a transparent executable reference. For final judging, install Qiskit and re-run until the Qiskit cell reports success.


In [1]:
# Run this cell once in a fresh environment, then restart the kernel.
# It is intentionally excluded from automated execution in this saved notebook.
%pip install "qiskit[visualization]==2.5.2" "qiskit-aer==0.17.2" pandas numpy scipy matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import itertools
import platform
import time
import warnings
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize

warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=5, suppress=True)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 140)

COLORS = {
    "navy": "#0B3954",
    "blue": "#087E8B",
    "aqua": "#5BC0BE",
    "gold": "#FFB703",
    "coral": "#FF5A5F",
    "gray": "#7A8894",
    "light": "#EAF4F4",
}

try:
    import qiskit
    from qiskit.circuit.library import qaoa_ansatz
    from qiskit.primitives import StatevectorEstimator
    from qiskit.quantum_info import SparsePauliOp, Statevector
    HAS_QISKIT = True
    QISKIT_VERSION = qiskit.__version__
except Exception as exc:
    HAS_QISKIT = False
    QISKIT_VERSION = f"not installed ({type(exc).__name__})"

print(f"Python {platform.python_version()} | NumPy {np.__version__} | pandas {pd.__version__} | SciPy {scipy.__version__}")
print(f"Qiskit: {QISKIT_VERSION}")


Python 3.14.8 | NumPy 2.5.4 | pandas 3.0.6 | SciPy 1.18.1
Qiskit: 2.5.2


## 2 · Inputs and provenance

The six candidates were selected to create a legible demonstration spanning westside, central, south, and north–south corridors. Average weekday boardings come from LA Metro's **Q2 CY2024 Line Level Ridership** attachment. Dividing by a fixed 25,000-boardings reference level produces a bounded efficiency score without exaggerating differences through sample-specific min–max scaling.

The production equity pipeline will calculate a route score from the share of stops or buffered route area intersecting high-need tracts in Metro's 2025 Equity Need Index (MENI), with ACS indicators retained for audit. The production overlap matrix will use Jaccard similarity:

$$O_{ij}=\frac{|S_i\cap S_j|}{|S_i\cup S_j|},$$

where $S_i$ is the set of stop catchments or census tracts served by route $i$.

Until that spatial work is complete, the equity scores and overlaps below are **prototype inputs** chosen only to test model behavior. The `data_status` fields make that boundary machine-readable.


In [3]:
routes = pd.DataFrame({
    "route": ["4", "18", "33", "40", "51", "204"],
    "corridor": [
        "Santa Monica–Downtown LA via Santa Monica Blvd",
        "Wilshire/Western–Montebello via 6th/Whittier",
        "Santa Monica–Downtown LA via Venice Blvd",
        "South Bay–Downtown LA via Hawthorne/MLK",
        "CSU Dominguez Hills–Westlake via Avalon/San Pedro",
        "Athens–Hollywood via Vermont Ave",
    ],
    "weekday_boardings_q2_2024": [24000, 21143, 17003, 14635, 18492, 22889],
    "equity_score": [0.44, 0.77, 0.58, 0.86, 0.93, 0.90],
})
routes["ridership_score"] = routes["weekday_boardings_q2_2024"] / 25000.0
routes["ridership_status"] = "observed: LA Metro Q2 CY2024"
routes["equity_status"] = "prototype: replace with GTFS × MENI"

ROUTE_IDS = routes["route"].tolist()
N = len(routes)
R = routes["ridership_score"].to_numpy(float)
E = routes["equity_score"].to_numpy(float)

display_columns = [
    "route", "corridor", "weekday_boardings_q2_2024",
    "ridership_score", "equity_score", "equity_status"
]
print(routes[display_columns].round(3).to_string(index=False))


route                                          corridor  weekday_boardings_q2_2024  ridership_score  equity_score                       equity_status
    4    Santa Monica–Downtown LA via Santa Monica Blvd                      24000            0.960          0.44 prototype: replace with GTFS × MENI
   18      Wilshire/Western–Montebello via 6th/Whittier                      21143            0.846          0.77 prototype: replace with GTFS × MENI
   33          Santa Monica–Downtown LA via Venice Blvd                      17003            0.680          0.58 prototype: replace with GTFS × MENI
   40           South Bay–Downtown LA via Hawthorne/MLK                      14635            0.585          0.86 prototype: replace with GTFS × MENI
   51 CSU Dominguez Hills–Westlake via Avalon/San Pedro                      18492            0.740          0.93 prototype: replace with GTFS × MENI
  204                  Athens–Hollywood via Vermont Ave                      22889            0.916 

In [4]:
# Prototype pairwise route-overlap inputs. Symmetric, zero diagonal, range [0, 1].
OVERLAP = np.array([
    [0.00, 0.08, 0.18, 0.02, 0.03, 0.06],
    [0.08, 0.00, 0.06, 0.03, 0.07, 0.10],
    [0.18, 0.06, 0.00, 0.05, 0.06, 0.05],
    [0.02, 0.03, 0.05, 0.00, 0.16, 0.12],
    [0.03, 0.07, 0.06, 0.16, 0.00, 0.14],
    [0.06, 0.10, 0.05, 0.12, 0.14, 0.00],
])
assert np.allclose(OVERLAP, OVERLAP.T)
assert np.allclose(np.diag(OVERLAP), 0)

ALPHA = 0.55       # equity weight; efficiency receives 1 - ALPHA
RHO = 0.35         # penalty for overlapping service
K = 3              # routes to fund
PENALTY = 1.50     # equality-constraint penalty

utility = (1 - ALPHA) * R + ALPHA * E
routes["blended_utility"] = utility

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
scatter = ax.scatter(R, E, s=170, c=utility, cmap="viridis", edgecolor="white", linewidth=1.5)
for i, route in enumerate(ROUTE_IDS):
    ax.annotate(f"Line {route}", (R[i], E[i]), xytext=(7, 6), textcoords="offset points", fontsize=9)
ax.axvline(R.mean(), color=COLORS["gray"], lw=1, ls="--")
ax.axhline(E.mean(), color=COLORS["gray"], lw=1, ls="--")
ax.set(xlabel="Ridership score (observed)", ylabel="Equity score (prototype)",
       title=f"Candidate routes · main equity weight α={ALPHA:.2f}")
ax.grid(alpha=.18)
colorbar = fig.colorbar(scatter, ax=ax, fraction=.046, pad=.04)
colorbar.set_label("Blended route utility")

im = axes[1].imshow(OVERLAP, cmap="Blues", vmin=0, vmax=OVERLAP.max())
axes[1].set_xticks(range(N), [f"{r}" for r in ROUTE_IDS])
axes[1].set_yticks(range(N), [f"{r}" for r in ROUTE_IDS])
axes[1].set_xlabel("Route")
axes[1].set_ylabel("Route")
axes[1].set_title("Prototype overlap matrix $O_{ij}$")
for i in range(N):
    for j in range(N):
        axes[1].text(j, i, f"{OVERLAP[i,j]:.2f}", ha="center", va="center", fontsize=8,
                     color="white" if OVERLAP[i,j] > .11 else COLORS["navy"])
fig.colorbar(im, ax=axes[1], fraction=.046, pad=.04)
plt.tight_layout()
plt.show()


/var/folders/_f/xl789qdd2sb3z0hwdw86qzvw0000gn/T/ipykernel_88825/1053282533.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Input interpretation

- **High ridership is not the same as high equity.** Line 4 has the largest observed ridership signal in this candidate set, while the prototype equity inputs emphasize Lines 40, 51, and 204.
- **The interaction term matters.** Funding two individually attractive but highly overlapping routes can deliver less portfolio coverage than funding complementary routes.
- **Every score is a policy choice.** Normalization, catchment size, demographic indicators, and weights must be published and reviewed—not hidden inside the optimizer.


## 3 · Optimization model

For each route $i$, let $x_i=1$ if it receives funding and $0$ otherwise. The route-level value under equity weight $\alpha$ is

$$u_i(\alpha)=(1-\alpha)R_i+\alpha E_i.$$

The portfolio benefit rewards route value and discounts redundant coverage:

$$F(x;\alpha)=\sum_i u_i(\alpha)x_i-\rho\sum_{i<j}O_{ij}x_ix_j.$$

Exactly $K=3$ routes must be funded. QAOA minimizes the penalized cost

$$C(x)=-F(x;\alpha)+P\left(\sum_i x_i-K\right)^2.$$

Here $\alpha=0$ means ridership-only, $\alpha=1$ means equity-only, $\rho$ controls overlap aversion, and $P$ enforces feasibility. The pairwise overlap term makes this a genuine portfolio problem rather than a simple top-three ranking.


In [ ]:
STATE_BITS = np.array(
    [[(state >> i) & 1 for i in range(N)] for state in range(2**N)],
    dtype=int,
)

def route_value(alpha):
    return (1 - alpha) * R + alpha * E

def portfolio_benefit(x, alpha=ALPHA, rho=RHO):
    x = np.asarray(x, dtype=float)
    individual = float(route_value(alpha) @ x)
    redundancy = sum(OVERLAP[i, j] * x[i] * x[j]
                     for i in range(N) for j in range(i + 1, N))
    return individual - rho * redundancy

def penalized_cost(x, alpha=ALPHA, rho=RHO, penalty=PENALTY, k=K):
    x = np.asarray(x, dtype=float)
    return -portfolio_benefit(x, alpha, rho) + penalty * (x.sum() - k) ** 2

def selected_routes(x):
    return [ROUTE_IDS[i] for i, bit in enumerate(x) if bit]

def bit_label(x):
    return "".join(str(int(v)) for v in x)

classical_start = time.perf_counter()
all_results = []
for state, x in enumerate(STATE_BITS):
    all_results.append({
        "state": state,
        "bits_route_order": bit_label(x),
        "selected": ", ".join(selected_routes(x)) or "none",
        "n_selected": int(x.sum()),
        "benefit": portfolio_benefit(x),
        "cost": penalized_cost(x),
        "feasible": bool(x.sum() == K),
    })
all_results = pd.DataFrame(all_results)
feasible_results = all_results.query("feasible").sort_values("benefit", ascending=False).reset_index(drop=True)
exact = feasible_results.iloc[0]
EXACT_STATE = int(exact["state"])
EXACT_BITS = STATE_BITS[EXACT_STATE]
EXACT_BENEFIT = float(exact["benefit"])
CLASSICAL_RUNTIME_S = time.perf_counter() - classical_start

print(f"Feasible combinations checked: {len(feasible_results)} = C(6, 3)")
print(f"Exact optimum: Lines {exact['selected']} | overlap-adjusted benefit = {EXACT_BENEFIT:.6f}")
print(f"Exhaustive-search runtime on this host: {CLASSICAL_RUNTIME_S:.6f} s")
print("\nTop five feasible portfolios:")
print(feasible_results[["selected", "benefit", "cost"]].head().round(6).to_string(index=False))


SyntaxError: invalid syntax. Perhaps you forgot a comma? (441044423.py, line 10)

### Why the exact solver comes first

For six routes there are only $\binom{6}{3}=20$ feasible portfolios, so exhaustive search is the correct benchmark. The classical result is the ground truth for judging QAOA solution quality. Runtime comparisons at this scale are descriptive only: brute force is expected to win.


## 4 · QUBO and Ising Hamiltonian

Expanding the penalty gives a QUBO of the form

$$C(x)=c_0+\sum_i a_ix_i+\sum_{i<j}b_{ij}x_ix_j,$$

with

$$c_0=PK^2,\qquad a_i=-u_i+P(1-2K),\qquad b_{ij}=\rho O_{ij}+2P.$$

Using $x_i=(I-Z_i)/2$ gives

$$H=cI+\sum_i h_iZ_i+\sum_{i<j}J_{ij}Z_iZ_j,$$

where

$$c=c_0+\frac12\sum_i a_i+\frac14\sum_{i<j}b_{ij},$$

$$h_i=-\frac{a_i}{2}-\frac14\sum_{j\ne i}b_{ij},\qquad J_{ij}=\frac{b_{ij}}4.$$

The next cell checks this conversion for **every one of the 64 computational-basis states**. This catches sign, offset, and qubit-order mistakes before QAOA is run.


In [ ]:
def qubo_coefficients(alpha=ALPHA, rho=RHO, penalty=PENALTY, k=K):
    u = route_value(alpha)
    c0 = penalty * k**2
    a = -u + penalty * (1 - 2 * k)
    b = np.zeros((N, N), dtype=float)
    for i in range(N):
        for j in range(i + 1, N):
            b[i, j] = rho * OVERLAP[i, j] + 2 * penalty
    return c0, a, b

def ising_coefficients(alpha=ALPHA, rho=RHO, penalty=PENALTY, k=K):
    c0, a, b = qubo_coefficients(alpha, rho, penalty, k)
    c = c0 + 0.5 * a.sum() + 0.25 * np.triu(b, 1).sum()
    h = np.empty(N, dtype=float)
    for i in range(N):
        incident = sum(b[min(i, j), max(i, j)] for j in range(N) if j != i)
        h[i] = -0.5 * a[i] - 0.25 * incident
    J = b / 4.0
    return c, h, J

def qubo_energy(x, alpha=ALPHA, rho=RHO, penalty=PENALTY, k=K):
    c0, a, b = qubo_coefficients(alpha, rho, penalty, k)
    x = np.asarray(x, dtype=float)
    return c0 + a @ x + sum(b[i, j] * x[i] * x[j]
                              for i in range(N) for j in range(i + 1, N))

def ising_energy(x, alpha=ALPHA, rho=RHO, penalty=PENALTY, k=K):
    c, h, J = ising_coefficients(alpha, rho, penalty, k)
    z = 1 - 2 * np.asarray(x, dtype=float)
    return c + h @ z + sum(J[i, j] * z[i] * z[j]
                           for i in range(N) for j in range(i + 1, N))

direct = np.array([penalized_cost(x) for x in STATE_BITS])
qubo = np.array([qubo_energy(x) for x in STATE_BITS])
ising = np.array([ising_energy(x) for x in STATE_BITS])
max_direct_qubo = float(np.max(np.abs(direct - qubo)))
max_qubo_ising = float(np.max(np.abs(qubo - ising)))

assert max_direct_qubo < 1e-10
assert max_qubo_ising < 1e-10
print("✓ Direct objective = expanded QUBO for all 64 states")
print("✓ QUBO = Ising basis energy for all 64 states")
print(f"Maximum numerical error: {max(max_direct_qubo, max_qubo_ising):.3e}")

COST_ENERGIES = ising.copy()
c_ising, h_ising, J_ising = ising_coefficients()
print(f"\nHamiltonian terms: 1 constant + {N} Z + {N*(N-1)//2} ZZ")
print(f"Constant offset c = {c_ising:.6f}")


✓ Direct objective = expanded QUBO for all 64 states
✓ QUBO = Ising basis energy for all 64 states
Maximum numerical error: 9.770e-15

Hamiltonian terms: 1 constant + 6 Z + 15 ZZ
Constant offset c = 0.060417


In [ ]:
def sparse_pauli_hamiltonian(c, h, J):
    # Build H using Qiskit's little-endian qubit convention (route i ↔ qubit i).
    terms = [("I" * N, float(c))]
    for i in range(N):
        label = ["I"] * N
        label[N - 1 - i] = "Z"
        terms.append(("".join(label), float(h[i])))
    for i in range(N):
        for j in range(i + 1, N):
            label = ["I"] * N
            label[N - 1 - i] = "Z"
            label[N - 1 - j] = "Z"
            terms.append(("".join(label), float(J[i, j])))
    return SparsePauliOp.from_list(terms).simplify()

if HAS_QISKIT:
    COST_HAMILTONIAN = sparse_pauli_hamiltonian(c_ising, h_ising, J_ising)
    qiskit_diagonal = np.real(COST_HAMILTONIAN.to_matrix().diagonal())
    assert np.max(np.abs(qiskit_diagonal - COST_ENERGIES)) < 1e-10
    print("✓ Qiskit SparsePauliOp diagonal matches all 64 expected energies")
    print(COST_HAMILTONIAN)
else:
    COST_HAMILTONIAN = None
    print("Qiskit is unavailable in this host; the verified coefficients above remain executable.")
    print("Install Qiskit with the setup cell, restart, and re-run before final submission.")


✓ Qiskit SparsePauliOp diagonal matches all 64 expected energies
SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIZII', 'IIZIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ', 'IZIIIZ', 'ZIIIIZ', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[0.06042+0.j, 0.30462+0.j, 0.37229+0.j, 0.27753+0.j, 0.33496+0.j,
 0.38193+0.j, 0.41238+0.j, 0.757  +0.j, 0.76575+0.j, 0.75175+0.j,
 0.75262+0.j, 0.75525+0.j, 0.75525+0.j, 0.75262+0.j, 0.75613+0.j,
 0.75875+0.j, 0.75438+0.j, 0.75525+0.j, 0.75438+0.j, 0.764  +0.j,
 0.7605 +0.j, 0.76225+0.j])


## 5 · Is the constraint penalty strong enough?

A penalty that is too small lets an infeasible state beat every valid portfolio; a huge penalty can make the variational landscape unnecessarily steep. We enumerate all states across several values and identify the smallest sufficient penalty on a fine grid.


In [ ]:
penalty_rows = []
for p_test in [0.00, 0.25, 0.50, 0.65, 0.75, 1.00, 1.50, 2.00]:
    costs = np.array([penalized_cost(x, penalty=p_test) for x in STATE_BITS])
    winner = int(np.argmin(costs))
    penalty_rows.append({
        "P": p_test,
        "global_min_routes": ", ".join(selected_routes(STATE_BITS[winner])) or "none",
        "n_selected": int(STATE_BITS[winner].sum()),
        "feasible": bool(STATE_BITS[winner].sum() == K),
        "minimum_cost": costs[winner],
    })
penalty_table = pd.DataFrame(penalty_rows)

fine_grid = np.linspace(0, 2, 2001)
minimum_sufficient = next(
    p for p in fine_grid
    if STATE_BITS[np.argmin([penalized_cost(x, penalty=p) for x in STATE_BITS])].sum() == K
)

print(penalty_table.round(4).to_string(index=False))
print(f"\nSmallest sufficient P on a 0.001 grid: {minimum_sufficient:.3f}")
print(f"Chosen P = {PENALTY:.2f} → feasible ground state: {PENALTY > minimum_sufficient}")

fig, ax = plt.subplots(figsize=(8.5, 3.6))
colors = [COLORS["blue"] if ok else COLORS["coral"] for ok in penalty_table["feasible"]]
ax.bar(penalty_table["P"].astype(str), penalty_table["n_selected"], color=colors)
ax.axhline(K, color=COLORS["navy"], ls="--", lw=1.5, label=f"Required K={K}")
ax.set(xlabel="Penalty P", ylabel="Routes in global minimum", title="Constraint feasibility versus penalty strength")
ax.set_ylim(0, N + .5)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


   P      global_min_routes  n_selected  feasible  minimum_cost
0.00 4, 18, 33, 40, 51, 204           6     False       -4.1674
0.25        18, 40, 51, 204           4     False       -2.8249
0.50        18, 40, 51, 204           4     False       -2.5749
0.65            18, 51, 204           3      True       -2.4469
0.75            18, 51, 204           3      True       -2.4469
1.00            18, 51, 204           3      True       -2.4469
1.50            18, 51, 204           3      True       -2.4469
2.00            18, 51, 204           3      True       -2.4469

Smallest sufficient P on a 0.001 grid: 0.628
Chosen P = 1.50 → feasible ground state: True


/var/folders/_f/xl789qdd2sb3z0hwdw86qzvw0000gn/T/ipykernel_88394/1194804828.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · QAOA

For depth $p$, QAOA alternates a cost unitary and a mixing unitary:

$$|\gamma,\beta\rangle=\prod_{\ell=1}^{p}e^{-i\beta_\ell\sum_iX_i}e^{-i\gamma_\ell H_C}|+\rangle^{\otimes n}.$$

A classical optimizer adjusts $(\gamma,\beta)$ to minimize $\langle H_C\rangle$. The reference implementation below exposes every step: diagonal phase evolution, single-qubit $X$ rotations, expectation calculation, and sampling. It is feasible only because this demonstration has six qubits; the Qiskit implementation follows immediately afterward.


In [ ]:
def apply_x_mixer(state, beta, n_qubits=N):
    state = state.copy()
    c, s = np.cos(beta), np.sin(beta)
    for q in range(n_qubits):
        stride = 1 << q
        for start in range(0, len(state), 2 * stride):
            for offset in range(stride):
                i0 = start + offset
                i1 = i0 + stride
                a0, a1 = state[i0], state[i1]
                state[i0] = c * a0 - 1j * s * a1
                state[i1] = -1j * s * a0 + c * a1
    return state

def qaoa_state(theta, cost_energies=COST_ENERGIES, depth=3):
    theta = np.asarray(theta, dtype=float)
    gammas, betas = theta[:depth], theta[depth:]
    state = np.ones(2**N, dtype=complex) / np.sqrt(2**N)
    for gamma, beta in zip(gammas, betas):
        state *= np.exp(-1j * gamma * cost_energies)
        state = apply_x_mixer(state, beta)
    return state

def qaoa_expectation(theta, cost_energies=COST_ENERGIES, depth=3):
    state = qaoa_state(theta, cost_energies, depth)
    probabilities = np.abs(state) ** 2
    return float(probabilities @ cost_energies)

def optimize_reference_qaoa(depth, seed, maxiter=700, cost_energies=COST_ENERGIES):
    rng = np.random.default_rng(seed)
    x0 = np.concatenate([
        rng.uniform(-np.pi, np.pi, depth),
        rng.uniform(-np.pi / 2, np.pi / 2, depth),
    ])
    history = []
    def objective(theta):
        value = qaoa_expectation(theta, cost_energies, depth)
        history.append(value)
        return value
    start = time.perf_counter()
    result = minimize(objective, x0, method="COBYLA", options={"maxiter": maxiter, "tol": 1e-7})
    elapsed = time.perf_counter() - start
    state = qaoa_state(result.x, cost_energies, depth)
    probabilities = np.abs(state) ** 2
    return {"depth": depth, "seed": seed, "result": result, "history": history,
            "probabilities": probabilities, "runtime_s": elapsed}

def probability_metrics(probabilities):
    feasible_mask = STATE_BITS.sum(axis=1) == K
    feasible_probability = float(probabilities[feasible_mask].sum())
    optimum_probability = float(probabilities[EXACT_STATE])
    feasible_states = np.flatnonzero(feasible_mask)
    modal_feasible_state = int(feasible_states[np.argmax(probabilities[feasible_states])])
    benefits = np.array([portfolio_benefit(x) for x in STATE_BITS])
    conditional_expected_benefit = float(
        probabilities[feasible_mask] @ benefits[feasible_mask] / feasible_probability
    )
    return {
        "feasibility_probability": feasible_probability,
        "optimum_probability": optimum_probability,
        "modal_feasible_state": modal_feasible_state,
        "conditional_expected_benefit": conditional_expected_benefit,
    }

# Unit checks for norm preservation and expectation bounds.
test_state = qaoa_state(np.zeros(6), depth=3)
assert np.isclose(np.linalg.norm(test_state), 1.0)
assert COST_ENERGIES.min() <= qaoa_expectation(np.zeros(6), depth=3) <= COST_ENERGIES.max()
print("✓ Reference QAOA state remains normalized")


✓ Reference QAOA state remains normalized


In [ ]:
MAIN_DEPTH = 3
MAIN_SEEDS = range(12)

main_runs = [optimize_reference_qaoa(MAIN_DEPTH, seed, maxiter=700) for seed in MAIN_SEEDS]
for run in main_runs:
    run.update(probability_metrics(run["probabilities"]))

best_run = min(main_runs, key=lambda item: item["result"].fun)
BEST_PROBS = best_run["probabilities"]
best_metrics = probability_metrics(BEST_PROBS)

run_table = pd.DataFrame([{
    "seed": run["seed"],
    "expected_cost": run["result"].fun,
    "feasible_probability": run["feasibility_probability"],
    "optimum_probability": run["optimum_probability"],
    "modal_feasible": ", ".join(selected_routes(STATE_BITS[run["modal_feasible_state"]])),
    "iterations": run["result"].nfev,
    "runtime_s": run["runtime_s"],
} for run in main_runs]).sort_values("expected_cost")

print(f"Best of {len(main_runs)} random starts: seed {best_run['seed']}")
print(run_table.round(4).to_string(index=False))


Best of 12 random starts: seed 10
 seed  expected_cost  feasible_probability  optimum_probability modal_feasible  iterations  runtime_s
   10        -2.1380                0.9697               0.0875    18, 51, 204         700     0.5231
    5        -2.0677                0.9540               0.0339      4, 33, 40         700     0.5585
    6        -2.0469                0.9246               0.0477    18, 51, 204         700     0.5231
    8        -1.9681                0.8046               0.1250    18, 51, 204         700     0.5172
   11        -1.8374                0.7428               0.0614    18, 51, 204         700     0.5376
    3        -1.7903                0.6262               0.0604    18, 51, 204         700     0.5163
    4        -1.6773                0.7722               0.0365      4, 33, 40         696     0.5701
    2        -1.6353                0.8809               0.0542    18, 51, 204         700     0.5136
    0        -1.2796                0.2751      

In [ ]:
# Draw finite shots from the best statevector so the reported bitstring is measurement-like.
SHOTS = 4096
sample_rng = np.random.default_rng(2026)
sampled_states = sample_rng.choice(2**N, size=SHOTS, p=BEST_PROBS)
counts = Counter(sampled_states)
feasible_sample_counts = {s: count for s, count in counts.items() if STATE_BITS[s].sum() == K}
SAMPLED_STATE = max(feasible_sample_counts, key=feasible_sample_counts.get)
SAMPLED_BITS = STATE_BITS[SAMPLED_STATE]
SAMPLED_BENEFIT = portfolio_benefit(SAMPLED_BITS)

top_states = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:12]
top_table = pd.DataFrame([{
    "route-order bits": bit_label(STATE_BITS[state]),
    "selected routes": ", ".join(selected_routes(STATE_BITS[state])) or "none",
    "count": count,
    "probability": count / SHOTS,
    "feasible": bool(STATE_BITS[state].sum() == K),
    "exact optimum": state == EXACT_STATE,
} for state, count in top_states])

print(f"Sampled most-common feasible portfolio: Lines {', '.join(selected_routes(SAMPLED_BITS))}")
print(f"Constraint satisfied: {SAMPLED_BITS.sum() == K}")
print(f"Portfolio benefit: {SAMPLED_BENEFIT:.6f}")
print("\nMost frequent sampled states:")
print(top_table.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
axes[0].plot(best_run["history"], color=COLORS["blue"], lw=2)
axes[0].axhline(COST_ENERGIES.min(), color=COLORS["gold"], ls="--", label="Exact ground energy")
axes[0].set(xlabel="Objective evaluations", ylabel="Expected penalized cost",
            title=f"Best QAOA optimization trace · p={MAIN_DEPTH}, seed={best_run['seed']}")
axes[0].legend(frameon=False)
axes[0].grid(alpha=.2)

labels = [row["selected routes"] for _, row in top_table.iterrows()]
bar_colors = [COLORS["gold"] if row["exact optimum"] else
              (COLORS["blue"] if row["feasible"] else COLORS["coral"])
              for _, row in top_table.iterrows()]
axes[1].barh(range(len(top_table)), top_table["probability"], color=bar_colors)
axes[1].set_yticks(range(len(top_table)), labels, fontsize=8)
axes[1].invert_yaxis()
axes[1].set(xlabel="Sample frequency", title=f"Top measured portfolios · {SHOTS:,} shots")
axes[1].grid(axis="x", alpha=.2)
plt.tight_layout()
plt.show()

Sampled most-common feasible portfolio: Lines 18, 51, 204
Constraint satisfied: True
Portfolio benefit: 2.446932

Most frequent sampled states:
route-order bits selected routes  count  probability  feasible  exact optimum
          010011     18, 51, 204    338       0.0825      True           True
          010101     18, 40, 204    300       0.0732      True          False
          100011      4, 51, 204    268       0.0654      True          False
          000111     40, 51, 204    251       0.0613      True          False
          010110      18, 40, 51    250       0.0610      True          False
          110001      4, 18, 204    231       0.0564      True          False
          001011     33, 51, 204    228       0.0557      True          False
          100101      4, 40, 204    216       0.0527      True          False
          110010       4, 18, 51    200       0.0488      True          False
          011001     18, 33, 204    200       0.0488      True          Fals

/var/folders/_f/xl789qdd2sb3z0hwdw86qzvw0000gn/T/ipykernel_88394/2276655392.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
comparison = pd.DataFrame([
    {
        "method": "Exact brute force",
        "selected_routes": ", ".join(selected_routes(EXACT_BITS)),
        "benefit": EXACT_BENEFIT,
        "feasible": True,
        "optimality_ratio": 1.0,
        "success_probability": 1.0,
        "runtime_s": CLASSICAL_RUNTIME_S,
    },
    {
        "method": f"QAOA p={MAIN_DEPTH} (modal feasible sample)",
        "selected_routes": ", ".join(selected_routes(SAMPLED_BITS)),
        "benefit": SAMPLED_BENEFIT,
        "feasible": bool(SAMPLED_BITS.sum() == K),
        "optimality_ratio": SAMPLED_BENEFIT / EXACT_BENEFIT,
        "success_probability": best_metrics["optimum_probability"],
        "runtime_s": best_run["runtime_s"],
    },
])

print(comparison.round(4).to_string(index=False))
print(f"\nBest-state feasibility probability: {best_metrics['feasibility_probability']:.2%}")
print(f"Exact-optimum probability: {best_metrics['optimum_probability']:.2%}")
print(f"Expected benefit conditioned on feasibility: {best_metrics['conditional_expected_benefit']:.6f}")
print(f"Random starts whose modal feasible state is exact: "
      f"{sum(run['modal_feasible_state'] == EXACT_STATE for run in main_runs)}/{len(main_runs)}")


                          method selected_routes  benefit  feasible  optimality_ratio  success_probability  runtime_s
               Exact brute force     18, 51, 204   2.4469      True               1.0               1.0000     0.0093
QAOA p=3 (modal feasible sample)     18, 51, 204   2.4469      True               1.0               0.0875     0.5231

Best-state feasibility probability: 96.97%
Exact-optimum probability: 8.75%
Expected benefit conditioned on feasibility: 2.248047
Random starts whose modal feasible state is exact: 6/12


### Reading the QAOA result

The modal feasible sample is the operational answer; “best observed” is not silently substituted for it. The notebook also reports feasibility probability, probability of the exact optimum, conditional expected benefit, and variability across starts. These metrics distinguish solution quality from a lucky rare sample.

The small classical problem remains easy. QAOA is valuable here as a correct, inspectable quantum formulation—not as evidence of speedup.


## 7 · Qiskit implementation

This is the submission-facing Qiskit path. It consumes the already verified `SparsePauliOp`, builds the standard QAOA ansatz, minimizes its expectation with a statevector estimator, and decodes probabilities using the same route-to-qubit convention. The cell falls back cleanly when Qiskit is absent; a final submission run should show the success line.


In [ ]:
QISKIT_RUN = None

if HAS_QISKIT:
    qiskit_depth = 3
    ansatz = qaoa_ansatz(COST_HAMILTONIAN, reps=qiskit_depth, flatten=True)
    estimator = StatevectorEstimator(seed=2026)
    qiskit_history = []

    def qiskit_objective(theta):
        pub_result = estimator.run([(ansatz, COST_HAMILTONIAN, theta)]).result()[0]
        value = float(np.real(pub_result.data.evs))
        qiskit_history.append(value)
        return value

    rng = np.random.default_rng(2026)
    theta0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
    start = time.perf_counter()
    qiskit_result = minimize(
        qiskit_objective, theta0, method="COBYLA",
        options={"maxiter": 700, "tol": 1e-7},
    )
    qiskit_runtime = time.perf_counter() - start
    bound = ansatz.assign_parameters(qiskit_result.x)
    qiskit_probabilities = Statevector.from_instruction(bound).probabilities()
    qiskit_metrics = probability_metrics(qiskit_probabilities)
    QISKIT_RUN = {
        "result": qiskit_result,
        "runtime_s": qiskit_runtime,
        "probabilities": qiskit_probabilities,
        **qiskit_metrics,
    }
    qiskit_modal = STATE_BITS[qiskit_metrics["modal_feasible_state"]]
    print("✓ Qiskit QAOA completed")
    print(f"Modal feasible portfolio: Lines {', '.join(selected_routes(qiskit_modal))}")
    print(f"Feasibility probability: {qiskit_metrics['feasibility_probability']:.2%}")
    print(f"Exact-optimum probability: {qiskit_metrics['optimum_probability']:.2%}")
else:
    print("SKIPPED: Qiskit is not installed in this execution host.")
    print("Required before submission: run the install cell, restart, and confirm this cell prints ✓.")


✓ Qiskit QAOA completed
Modal feasible portfolio: Lines 4, 33, 40
Feasibility probability: 95.10%
Exact-optimum probability: 4.76%


## 8 · Robustness experiments

One successful run is not enough. We next ask whether performance changes with circuit depth and seed, whether the policy answer changes with $\alpha$, and whether the overlap interaction changes the recommendation.


In [ ]:
depth_runs = []
for depth in [1, 2]:
    for seed in range(5):
        run = optimize_reference_qaoa(depth, seed, maxiter=500)
        run.update(probability_metrics(run["probabilities"]))
        depth_runs.append(run)
depth_runs.extend(main_runs)

depth_summary = pd.DataFrame([{
    "depth": run["depth"],
    "seed": run["seed"],
    "expected_cost": run["result"].fun,
    "feasibility_probability": run["feasibility_probability"],
    "optimum_probability": run["optimum_probability"],
    "modal_exact": run["modal_feasible_state"] == EXACT_STATE,
} for run in depth_runs])

depth_aggregate = depth_summary.groupby("depth").agg(
    runs=("seed", "count"),
    median_expected_cost=("expected_cost", "median"),
    best_expected_cost=("expected_cost", "min"),
    median_feasibility=("feasibility_probability", "median"),
    best_optimum_probability=("optimum_probability", "max"),
    modal_exact_rate=("modal_exact", "mean"),
).reset_index()

print(depth_aggregate.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(depth_aggregate))
width = .34
ax.bar(x - width/2, depth_aggregate["median_feasibility"], width,
       color=COLORS["blue"], label="Median feasible probability")
ax.bar(x + width/2, depth_aggregate["best_optimum_probability"], width,
       color=COLORS["gold"], label="Best exact-optimum probability")
ax.set_xticks(x, [f"p={d}" for d in depth_aggregate["depth"]])
ax.set_ylim(0, 1)
ax.set(ylabel="Probability", title="QAOA depth and random-start sensitivity")
ax.legend(frameon=False, ncol=2)
ax.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.show()


 depth  runs  median_expected_cost  best_expected_cost  median_feasibility  best_optimum_probability  modal_exact_rate
     1     5               -0.0526             -1.4597              0.3238                    0.0306               0.2
     2     5               -1.6259             -1.9661              0.7212                    0.0398               0.4
     3    12               -1.7338             -2.1380              0.7575                    0.1250               0.5


/var/folders/_f/xl789qdd2sb3z0hwdw86qzvw0000gn/T/ipykernel_88394/631568157.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
alphas = np.linspace(0, 1, 21)
sweep_rows = []
selection_matrix = np.zeros((N, len(alphas)), dtype=int)

for col, alpha in enumerate(alphas):
    candidates = []
    for x in STATE_BITS[STATE_BITS.sum(axis=1) == K]:
        candidates.append((portfolio_benefit(x, alpha=alpha, rho=RHO), x.copy()))
    best_benefit, best_x = max(candidates, key=lambda pair: pair[0])
    selection_matrix[:, col] = best_x
    sweep_rows.append({
        "alpha": alpha,
        "selected_routes": ", ".join(selected_routes(best_x)),
        "portfolio_benefit": best_benefit,
        "ridership_component": float(R @ best_x),
        "equity_component": float(E @ best_x),
    })
sweep = pd.DataFrame(sweep_rows)

scenario_table = sweep.loc[sweep["selected_routes"].ne(sweep["selected_routes"].shift())].copy()
scenario_table["begins_at_alpha"] = scenario_table["alpha"]
print("Portfolio transition points on the 0.05 grid:")
print(scenario_table[["begins_at_alpha", "selected_routes", "ridership_component", "equity_component"]]
      .round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={"width_ratios": [1.2, 1]})
cmap = ListedColormap(["#EFF4F6", COLORS["gold"]])
axes[0].imshow(selection_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=1)
axes[0].set_yticks(range(N), [f"Line {r}" for r in ROUTE_IDS])
axes[0].set_xticks(range(0, len(alphas), 2), [f"{a:.1f}" for a in alphas[::2]])
axes[0].set(xlabel="Equity weight α", title="Exact portfolio changes as policy priorities change")

axes[1].plot(sweep["alpha"], sweep["ridership_component"], marker="o", ms=3,
             color=COLORS["blue"], label="Sum of ridership scores")
axes[1].plot(sweep["alpha"], sweep["equity_component"], marker="o", ms=3,
             color=COLORS["coral"], label="Sum of equity scores")
axes[1].axvline(ALPHA, color=COLORS["gold"], ls="--", label=f"Main α={ALPHA}")
axes[1].set(xlabel="Equity weight α", ylabel="Unweighted portfolio component",
            title="Who the objective rewards")
axes[1].legend(frameon=False)
axes[1].grid(alpha=.2)
plt.tight_layout()
plt.show()


Portfolio transition points on the 0.05 grid:
 begins_at_alpha selected_routes  ridership_component  equity_component
            0.00      4, 18, 204                2.721              2.11
            0.35     18, 51, 204                2.501              2.60
            0.90     40, 51, 204                2.241              2.69


/var/folders/_f/xl789qdd2sb3z0hwdw86qzvw0000gn/T/ipykernel_88394/3231788793.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Policy interpretation of the weight sweep

At low $\alpha$, the model emphasizes routes with the largest observed ridership. As $\alpha$ rises, Lines 51 and 40 become more competitive in the prototype scenario. These transition points are not discoveries about LA residents; they are an audit of how the **chosen objective** moves public resources. A deployment should present this frontier to decision-makers and community members rather than hiding it behind one default slider position.


In [ ]:
ablation_rows = []
for alpha in [0.00, 0.20, 0.40, 0.55, 0.70, 0.80, 0.90, 1.00]:
    row = {"alpha": alpha}
    for rho_value, label in [(0.0, "no_overlap_term"), (RHO, "with_overlap_term")]:
        choices = [
            (portfolio_benefit(x, alpha=alpha, rho=rho_value), x.copy())
            for x in STATE_BITS if x.sum() == K
        ]
        _, winner = max(choices, key=lambda pair: pair[0])
        row[label] = ", ".join(selected_routes(winner))
    row["recommendation_changes"] = row["no_overlap_term"] != row["with_overlap_term"]
    ablation_rows.append(row)
ablation = pd.DataFrame(ablation_rows)

print(ablation.to_string(index=False))
print(f"\nScenarios changed by overlap term: {ablation['recommendation_changes'].sum()}/{len(ablation)}")


 alpha no_overlap_term with_overlap_term  recommendation_changes
  0.00      4, 18, 204        4, 18, 204                   False
  0.20      4, 18, 204        4, 18, 204                   False
  0.40     18, 51, 204       18, 51, 204                   False
  0.55     18, 51, 204       18, 51, 204                   False
  0.70     18, 51, 204       18, 51, 204                   False
  0.80     40, 51, 204       18, 51, 204                    True
  0.90     40, 51, 204       40, 51, 204                   False
  1.00     40, 51, 204       40, 51, 204                   False

Scenarios changed by overlap term: 1/8


### What the robustness tests show

- Greater depth can concentrate more probability on feasible, high-quality states, but it also adds parameters and does not guarantee a better local optimum.
- Multiple seeds are necessary: variational optimization is stochastic and non-convex.
- The equity slider changes the selected portfolio, directly answering the hackathon's central values question.
- Removing pairwise overlap can change the recommendation near a transition point. This is the model's originality contribution and a reminder that portfolios are more than independent route rankings.


## 9 · Equity, harms, and governance

### Who may benefit in the main prototype scenario

The main solution favors Lines 18, 51, and 204 because their combined blended value remains strong after the overlap discount. In a completed analysis, the dashboard should report the neighborhoods, rider groups, times of day, and accessibility needs served by each selected and unselected route—not just route numbers.

### What this model can get wrong

| Risk | Why it matters | Mitigation before real use |
|---|---|---|
| Historic-ridership bias | Current boardings reflect past service levels and underinvestment; suppressed demand is invisible | Add unmet-demand/service-quality indicators and scenario-test them |
| Normative equity proxy | Poverty, vehicle access, disability, race, age, and transit dependence are not interchangeable | Co-design the index and weights; publish indicator-level results |
| Ecological fallacy | A route-average tract score does not describe every rider | Pair spatial data with anonymized rider surveys and disaggregated audits |
| ACS uncertainty | Survey estimates have margins of error and lag | Propagate uncertainty, display intervals, and refresh data |
| Route geometry approximation | Buffers and stop catchments can overstate who can access service | Test multiple catchments and use pedestrian-network access where possible |
| Displacement and policing | Service investment can raise land pressure or increase enforcement harms | Coordinate anti-displacement protections and prohibit enforcement-based “success” metrics |
| Optimization authority | A precise score can falsely appear objective | Keep the model advisory; document overrides, appeals, and public comments |

### Governance commitments

1. Publish raw indicators, transformations, weights, code, and rejected alternatives.  
2. Require community review before fixing $\alpha$ or the equity definition.  
3. Report outcomes for both selected and non-selected corridors.  
4. Add accessibility, span, reliability, and frequency constraints before operational use.  
5. Re-evaluate after investment and allow an appeal or correction process.


## 10 · Conclusion and submission checklist

### Main finding

For the prototype LA scenario with $\alpha=0.55$, exact enumeration selects **Lines 18, 51, and 204**. The QAOA experiment is judged against that known answer, and its probability of feasibility and optimality are reported rather than obscured. The weight sweep demonstrates the core societal result: changing the definition of value changes which corridors receive investment.

> **Takeaway for judges:** Quantum optimization can explore competing allocation priorities, but “optimal” is conditional on the values, data, and governance encoded in the objective.


## Sources, licenses, and attribution

### Data and technical references

1. LA Metro, **Line Level Ridership, Q2 CY2024**, Board Report 2024-0529, Attachment B: [PDF](https://metro.legistar1.com/metro/attachments/e204fb22-2c72-4a46-b95d-dd479e6612cf.pdf).  
2. LA Metro, **Maps & Schedules**: [metro.net/riding/schedules](https://www.metro.net/riding/schedules/).  
3. LA Metro, **Bus GTFS**: [GitLab feed](https://gitlab.com/LACMTA/gtfs_bus/raw/master/gtfs_bus.zip).  
4. LA Metro, **2025 Metro Equity Need Index**: [ArcGIS item](https://www.arcgis.com/home/item.html?id=c557e8b8fe1348c88a35a707612df3cb).  
5. U.S. Census Bureau, **2024 ACS 5-Year API**: [API documentation](https://api.census.gov/data/2024/acs/acs5.html).  
6. IBM Quantum, **Quantum Approximate Optimization Algorithm tutorial**: [Qiskit tutorial](https://quantum.cloud.ibm.com/docs/tutorials/quantum-approximate-optimization-algorithm).  
7. Qompute 2026: [hackathon repository](https://github.com/sandraalhames/Qompute-2026/).

### Template attribution

The pedagogical progression—problem, Hamiltonian, QAOA, sampling, and exact comparison—was adapted from the supplied **Track 2 Challenge 2 Tutorial** by Benjamin Tirado / BasQ. As marked in that source notebook, its narrative content is licensed under **CC BY 4.0** and its code under **Apache License 2.0**. All LA-specific modeling, validation, experiments, and impact analysis in this notebook were newly developed for QuantumTransit LA.

### Responsible-use note

This notebook is an educational prototype, not a recommendation to change LA Metro service. Prototype equity and overlap inputs must not be used for policy decisions.
